# Dynamic Vision Sensors (DVS)
A not-so-recent technology developed to mimic the humain eye has been making strides in resource-intensive fields like robotics, manufacturing, among others. Just like the humain brain, these eye-mimicing sensors are power efficient and only use the data they need to detect what they need.

Lichtsteiner et al. worked on designing and engineering a sensor that could do just that, eventually being the core technology behind many commercially successful sensors like iniVation's [DAVIS346](https://shop.inivation.com/products/davis346?variant=31028381188150).

The design mainly is building a sensor with three main parts:
- Logarithmic Photoreceptor - detects light and uses logarithm to put it within an easier range to work with
- Differencing circuit - stores last event's voltage to compare with new one for further processing
- Comparator - determine if voltage change is big enough according to a predefined threshold
- Async communication logic - sends pixel address when change is detected

## Building a simplistic model of a DVS

#### Logarithmic photoreceptor

In [ ]:
def log_photoreceptor(I):
	return np.log(I + 1e-9) # to avoid 0s

#### Differencing circuit
Here, `gain` scales the result of the difference since the change in voltage can be very small (in mV). This very tiny difference in voltage is one reason why neuromorphic sensors are power-efficient and can work on edge devices.

In [31]:
def differencing_circuit(last_voltage, new_voltage, gain=10): # gain being approx. C1 / C2
	return gain * (last_voltage - new_voltage) # multiplying gain acts as the sort of thing an amplifier would do

### Comparator
This is when you decide which events to include through your choice of change in light (threshold). In other words, the comparator is a filtering process that tries to answer the question 'Is this enough voltage difference to care about?'

This helps:
- Avoid data noise
- Lessen power load

In [29]:
def comparator(voltage_diff, threshold):
    if voltage_diff > threshold:  # when scene brightens
        return "ON"
    elif voltage_diff < -threshold:  # when scene dims, i.e. when change in last two events is less than the negative of your threshold
        return "OFF"
    else:
        return None

# or if you prefer a more readable code
def readable_comparator(voltage_diff, threshold):
    if abs(voltage_diff) < threshold:
        return None
    return "ON" if voltage_diff > 0 else "OFF"

### Simulating how brightness would be processed

In [13]:
brightness = np.array([100, 102, 105, 120, 125, 126, 80, 70, 69, 68])
I_photo = brightness  # assume photocurrent is proportional to brightness

events = []
V_prev = None
threshold = 0.3  # try between 0.1–0.5 depending on how strict you want
gain = 5

for t in range(len(I_photo)):
    V = log_photoreceptor(I_photo[t])
    if V_prev is None:
        V_prev = V
        continue

    diff = differencing_circuit(V, V_prev, gain=gain)
    event = readable_comparator(diff, threshold)

    if event:
        events.append((t, event, diff))
        V_prev = V  # reset after event, just like the circuit!

for e in events:
    print(f"t={e[0]}, {e[1]} event, ΔV amplified = {e[2]:.3f}")


t=3, ON event, ΔV amplified = 0.912
t=6, OFF event, ΔV amplified = -2.027
t=7, OFF event, ΔV amplified = -0.668


## Previous attempts towards event-based sensors
A problem consistent with solutions prior to one proposed by the paper boils down to how there's a mismatch between pixels (i.e. baseline voltage is not uniform across pixels) caused by manufacturing variations. One can say that it's a design issue: choosing a transistor to act as a resistor (whose effective resistance isn’t fixed) makes it impossible to set a low threshold for the array.

A quick look at prior solutions
1. Mahowald & Mead's silicon model of the human eye - refer to earlier notes in the `01-bio-inspired-vision` folder. It is the foundational paper behind all silicon retina papers.
2. Zaghoul and Boahen
> This design comes closest to capturing key adaptive fea- tures of biological retinas. It is achieved by the use of small- transistor log-domain circuits that are tightly coupled spatially by diffuser networks. However, this circuit design style led to large mismatch: the pixel firing rates vary by a standard de- viation of 1–2 decades and more than half the pixels do not spike at all for stimuli with 50% contrast.
3.  CSEM Neuchatel's group
> The group at CSEM Neuchatel [5] presented a device that is closest to being dual in functionality to the one reported here in that its output encodes spatial rather than temporal contrast: After a global frame integration period, this device transmits events in the order of high-to-low spatial contrast. Thus, readout can be aborted early if limited processing time is avail- able without losing information about high-contrast features. Each contrast event is followed by another event that encodes gradient orientation. This device has low 2% contrast mismatch and a large 6 decade dynamic range. They are presently in commercial development for automotive applications [14]. The main limitation of this architecture is that it does not reduce temporal redundancy (compute temporal derivatives), and its temporal resolution is limited to the frame rate.
4. Etienne-Cumming
> Etienne-Cumming’s group reported a temporal change threshold detection imager [3], which modifies the traditional active pixel sensor (APS) CMOS pixel so that it can detect a quantized absolute change in illumination